### 一、 原始数据统计

#### 1.1
接下来想统计users.csv中用户都发了多少post（post.csv）和comment （comment.csv）,分别计数，
数量记为post_sum, comment_sum，按comment_sum大小排列,
相等时按照post_sum排序，打印出相关中间结果

In [ ]:
import pandas as pd
# 设置路径
sub_dataset = '30W_valid_user'
users_test_path = f'/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/{sub_dataset}/user.csv'
post_test_path = f'/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/{sub_dataset}/post.csv'
comment_test_path = f'/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/{sub_dataset}/comment.csv'
# 读取 users_test.csv，post_test.csv 和 comment_test.csv
users_test_df = pd.read_csv(users_test_path)
post_test_df = pd.read_csv(post_test_path)
comment_test_df = pd.read_csv(comment_test_path)

# 统计每个用户发的 post 数量
post_count = post_test_df['creator'].value_counts().reset_index()
post_count.columns = ['id', 'post_sum']

# 统计每个用户发的 comment 数量
comment_count = comment_test_df['creator'].value_counts().reset_index()
comment_count.columns = ['id', 'comment_sum']

# 合并 post_count 和 comment_count
user_post_comment_count = pd.merge(post_count, comment_count, on='id', how='outer').fillna(0)

# 将合并后的数据与 users_test_df 合并
final_df = pd.merge(users_test_df, user_post_comment_count, on='id', how='left')

# 按照 comment_sum 降序排序，如果 comment_sum 相等则按照 post_sum 排序
final_df_sorted = final_df.sort_values(by=['comment_sum', 'post_sum'], ascending=False)

# 输出排序后的相关中间结果
print("排序后的用户发帖和评论数量：")
print(final_df_sorted[['id', 'post_sum', 'comment_sum']])

# 如果需要保存结果，可以使用下面的代码：
final_df_sorted.to_csv('/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/sorted_user.csv', index=False)


In [ ]:
for _, row in final_df_sorted.iterrows():
    single_row_df = row.to_frame().T  # 转换为单行 DataFrame
    print('\n')


2.统计user的bio信息

In [ ]:
import pandas as pd
# 读入
df = pd.read_csv('/home/wangshuo/resource/datasets/parler/csv_data/sub_data/users_test1.csv', dtype=str)
# 计算空 bio 的掩码
empty_mask = df['bio'].isna() | df['bio'].str.strip().eq('')
# 计算每条 bio 的单词数
word_counts = df['bio'].fillna('').str.split().apply(len)
# 保留 bio 非空 且 单词数 >= 3
keep_mask = (~empty_mask) & (word_counts >= 3)
# 筛选
df_filtered = df[keep_mask].copy()
# 输出结果统计
print(f"原始行数：{len(df)}")
print(f"保留行数：{len(df_filtered)}")
# （可选）把筛选后的结果写回文件
df_filtered.to_csv('/home/wangshuo/resource/datasets/parler/csv_data/sub_data/users_test1.csv',
                   index=False)


3.输出数据文件中数据项的数量

In [ ]:
import pandas as pd

# 文件路径
data_dir='/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'
comments2_test_large_file = data_dir + 'comment.csv'
users2_test_large_file = data_dir+ 'user.csv'
post_test_large_file = data_dir + 'post.csv'


print("读取 comments2_test_large.csv 文件...")
comments2_test_large_df = pd.read_csv(comments2_test_large_file)
users2_test_large_df = pd.read_csv(users2_test_large_file)
post_test_large_df = pd.read_csv(post_test_large_file)
print(len(comments2_test_large_df), "条评论数据已加载。")
print(len(users2_test_large_df), "条用户数据已加载。")
print(len(post_test_large_df), "条帖子数据已加载。")


读取 comments2_test_large.csv 文件...
131268 条评论数据已加载。
48516 条用户数据已加载。
24592 条帖子数据已加载。

4.输出一个csv文件某一个属性各值的数量

In [ ]:
import pandas as pd

# 1. 读入原始 CSV
datadir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5'
# 把下面的路径改成你的实际文件名或完整路径
csv_path = datadir + "/comment_test.csv"
df = pd.read_csv(csv_path)

# 2. 用 value_counts 统计某列（假设列名为 'your_column'）各值频次
vc1 = df['ML2_oracle1_label'].value_counts()
vc2 = df['ML2_proxy1_label'].value_counts()
vc3 = df['ML2_proxy2_label'].value_counts()
vc4 = df['ML2_proxy3_label'].value_counts()
vc5 = df['ML2_proxy4_label'].value_counts()
# vc6 = df['ML2_proxy7_label'].value_counts()

print(vc1)
print(vc2)
print(vc3)
print(vc4)
print(vc5)
# print(vc6)


### 二、ML 谓词数据统计
统计oracle模型和proxy模型输出的相似度

#### 2.2 统计ML1谓词---post相关信息

2.2.1 输入post.csv文件以ML1_label属性为准，统计ML1_label=ML1_entailment总数量，并分别统计元组属性ML1_proxy2_probability>0.6,0.7,0.8,0.85 时ML1_label=ML1_entailment元组数量， 

ML1_probability 代替ML1_lable，对于ML1_probability > delta1 =  [0.4, 0.5,0.6,0.65, 0.7, 0.8, 0.85,0.9]  分别统计数量， 对于ML1_proxy1_probability> delta2 =  [0.4, 0.5,0.6,0.65, 0.7, 0.8, 0.85,0.9]  分别统计数量，同时统计在ML1_probability > delta1 且ML1_proxy1_probability> delta2 的数量有多少，也就是双重循环

In [ ]:
import pandas as pd

# 1. 读取文件（替换为实际路径）
datadir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'
df = pd.read_csv(datadir + "post.csv")
oracle_pobability = 'ML1_oracle2_probability'
proxy_pobability = 'ML1_proxy6b1_probability'
# 定义阈值列表
delta1 = [0.5,0.6,0.7, 0.8, 0.85, 0.9]   # 对 ML1_probability
delta2 = [0.2,0.3,0.4,0.45,0.5, 0.6, 0.65, 0.7, 0.8, 0.84,0.85, 0.86,0.87,0.88,0.9]   # 对 ML1_proxy1_probability

# 双重循环：同时满足两个阈值条件的数量
print(f"\n=== 双阈值组合统计 ({oracle_pobability} > d1 且 {proxy_pobability} > d2) ===")
for d1 in delta1:
    TP = df[df[oracle_pobability] > d1].shape[0]
    print(f'd1={d1}: {oracle_pobability} > {d1} 的数量: {TP}')
    for d2 in delta2:
        R = df[df[proxy_pobability] > d2].shape[0]
        RTP = df[
            (df[oracle_pobability] > d1) &
            (df[proxy_pobability] > d2)
        ].shape[0]
        if R != 0 and TP != 0 and RTP != 0:
            print(f" d2={d2}: {proxy_pobability} > {d2} 的数量: {R}, 同时满足两个条件的数量: {RTP}, precision={RTP*1.0/R}, recall={RTP*1.0/TP}")
        else:
            print(f" d2={d2}: {proxy_pobability} > {d2} 的数量: {R}, 同时满足两个条件的数量: {RTP}, precision=0, recall=0")


根据precision={RTP*1.0/R}, recall={RTP*1.0/TP} 计算F1值，并给出F1值降序排名，每个项有F1分数，ML1_probability，ML1_proxy1_probability ，precision，recall

In [2]:
import pandas as pd
# datadir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'
# datadir = '/home/wangshuo/resource/datasets/parler_data/dataset_test/csv_data/'
# datadir = '/home/wangshuo/resource/datasets/parler_data/dataset_test/csv_data/'
datadir = '/home/wangshuo/resource/datasets/amazon_data/amazon_extend/csv_data/'
# filename = 'comment_test'
filename = 'product'
# filename = 'review'
# filename = 'comment'
file = datadir + f"{filename}.csv"
# 指标列名
oracle_probability = 'ML3_oracle2_probability'
proxy_probability  = 'ML3_proxy2_probability'
def find_optimle_proxy_threshold(file,oracle_probability,proxy_probability,config_threshold = 0.3): 
    # 1. 读取文件（替换为实际路径）
    df = pd.read_csv(file)
    # 定义阈值列表
    delta1 = [0.5, 0.6, 0.7, 0.8, 0.85, 0.9]
    delta2 = [0.2, 0.3, 0.4, 0.45, 0.5, 0.6, 0.65, 0.7, 0.8, 0.84, 0.85, 0.86, 0.87, 0.88, 0.9,0.95]
    results = []
    for d1 in delta1:
        TP = df[df[oracle_probability] > d1].shape[0]
        for d2 in delta2:
            R = df[df[proxy_probability] > d2].shape[0]
            RTP = df[
                (df[oracle_probability] > d1) &
                (df[proxy_probability] > d2)
            ].shape[0]
    
            precision = RTP / R if R > 0 else 0.0
            recall    = RTP / TP if TP > 0 else 0.0
            f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    
            results.append({
                oracle_probability: d1,
                proxy_probability:  d2,
                'precision': precision,
                'recall':    recall,
                'F1':        f1,
                'res': R,
                'ture': RTP,
                'all_ture': TP
            })
    # 转成 DataFrame
    res_df = pd.DataFrame(results)
    
    # 1) F1 降序（不加额外筛选）
    f1_sorted = res_df.sort_values(by='F1', ascending=False).reset_index(drop=True)
    print("\n=== 按 F1 值降序排名（前 10 条） ===")
    print(f1_sorted.head(10).to_string(index=False, float_format='%.4f'))
    
    # 2) Precision 降序，但先剔除 recall < 0.3 的组合
    pre_threshold = config_threshold
    prec_filtered = res_df[res_df['recall'] >= pre_threshold]
    prec_sorted = prec_filtered.sort_values(by='precision', ascending=False).reset_index(drop=True)
    print(f"\n=== 按 Precision 值降序排名（recall >= {config_threshold}，前 10 条） ===")
    print(prec_sorted.head(50).to_string(index=False, float_format='%.4f'))
    
    # 3) Recall 降序，但先剔除 precision < 0.3 的组合
    rec_threshold = config_threshold
    rec_filtered = res_df[res_df['precision'] >= rec_threshold]
    rec_sorted = rec_filtered.sort_values(by='recall', ascending=False).reset_index(drop=True)
    print(f"\n=== 按 Recall 值降序排名（precision >= {config_threshold}，前 10 条） ===")
    print(rec_sorted.head(10).to_string(index=False, float_format='%.4f'))
find_optimle_proxy_threshold(file,oracle_probability,proxy_probability,config_threshold = 0.3)


=== 按 F1 值降序排名（前 10 条） ===
 ML3_oracle2_probability  ML3_proxy2_probability  precision  recall     F1  res  ture  all_ture
                  0.9000                  0.9000     0.8536  0.8203 0.8366  444   379       462
                  0.8500                  0.8700     0.8580  0.8139 0.8354  479   411       505
                  0.8500                  0.8500     0.8427  0.8277 0.8352  496   418       505
                  0.8500                  0.8400     0.8327  0.8376 0.8351  508   423       505
                  0.8500                  0.8600     0.8487  0.8218 0.8350  489   415       505
                  0.8500                  0.8800     0.8675  0.8040 0.8345  468   406       505
                  0.9000                  0.8800     0.8291  0.8398 0.8344  468   388       462
                  0.9000                  0.8700     0.8184  0.8485 0.8332  479   392       462
                  0.5000                  0.4000     0.7860  0.8860 0.8330  860   676       763
            

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_probability_distribution(file_path, prob_cols, bins=None):
    """
    统计并绘制概率分布折线图，支持多列对比。
    
    参数:
        file_path: CSV 文件路径
        prob_cols: 要统计的概率列名，可以是单个字符串或字符串列表
        bins: 分箱区间列表，默认为 [0, 0.2, 0.4, 0.6, 0.8, 1.0]
    """
    # 1. 读取数据
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"[错误] 读取文件失败: {e}")
        return

    # 统一转为列表处理，方便后续循环
    if isinstance(prob_cols, str):
        prob_cols = [prob_cols]

    # 2. 设置分箱
    if bins is None:
        bins = [0.0, 0.05, 0.1, 0.15, 0.2, 0.4, 0.6, 0.8, 1.0]
    
    # 生成标签 (例如 "0.0-0.2")
    labels = [f"{bins[i]:.2f}-{bins[i+1]:.2f}" for i in range(len(bins)-1)]

    # 3. 绘图准备
    plt.figure(figsize=(12, 6))
    
    print("\n=== 概率分布统计 ===")
    
    # 4. 循环处理每一列
    for col in prob_cols:
        if col not in df.columns:
            print(f"[警告] 列名 '{col}' 不存在，跳过。")
            continue

        # 分箱统计
        # include_lowest=True 确保 0.0 被包含在第一个区间
        bucket_col = pd.cut(df[col], bins=bins, labels=labels, include_lowest=True)
        
        # 统计每个区间的数量，并按区间顺序排序 (reindex 确保所有 bin 都有值，即使是 0)
        counts = bucket_col.value_counts().reindex(labels, fill_value=0)
        
        # 计算比例
        percentages = (counts / len(df)) * 100

        # 打印该列统计信息
        print(f"\n--- 列名: {col} ---")
        stats_df = pd.DataFrame({'Count': counts, 'Percentage (%)': percentages})
        print(stats_df)

        # 绘制折线
        plt.plot(labels, counts.values, marker='o', linestyle='-', linewidth=2, markersize=6, label=col)
        
        # 如果只有一列，在点上显示数值；多列时为了避免重叠不显示
        if len(prob_cols) == 1:
            for i, count in enumerate(counts.values):
                plt.text(i, count, str(count), ha='center', va='bottom', fontsize=9)

    # 5. 图表设置
    plt.title('Probability Distribution Comparison', fontsize=14)
    plt.xlabel('Probability Range', fontsize=12)
    plt.ylabel('Count', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.xticks(rotation=45) # 标签可能较长，稍微旋转
    plt.legend() # 显示图例
    
    plt.tight_layout()
    plt.show()

# ==========================================
# 使用示例
# ==========================================
# datadir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'
datadir = '/home/wangshuo/resource/datasets/parler_data/dataset_one/csv_data/'
filename = 'post'
file_path = datadir + f"{filename}.csv"

# 定义要对比的列 (可以是列表)
# proxy_cols = ['ML1_proxy2b1_probability', 'ML1_oracle1_probability', 'ML1_proxy6b1_probability',
#               'ML1_proxy1b1_probability','ML1_proxy1_probability','ML1_proxy2_probability',
#               'ML1_proxy3_probability','ML1_proxy4_probability','ML1_proxy5_probability'] 
proxy_cols = ['ML1_oracle1_probability','ML1_proxy1_probability','ML1_proxy2_probability','ML1_proxy5_probability'] 

# 自定义区间
custom_bins = [0.0,0.05,0.1,0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plot_probability_distribution(file_path, proxy_cols, bins=custom_bins)

在文件中统计ML1_probability>0.3,0.4,0.5,0.6,0.7 的元组中有多少元组的LLM_label的属性为deepseek_yes

In [ ]:
import pandas as pd

# 1. 读入数据
filename = 'comment_ML2_proxy1'
# filename = 'post_LLM_cleaned_6'
input_path = f'/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/{filename}.csv'
# input_path = 'D:/softwareResource/datasets/IOGS/30W_valid_user/post_LLM_cleaned_2.csv'
df = pd.read_csv(input_path)

# 2. 筛选 ML1_probability > 0.3 且 LLM_label = 'deepseek_no'
mask_no = (df['ML1_oracle2_probability'] > 0.5) & (df['LLM_label'] == 'deepseek_yes')
df_no = df.loc[mask_no, 'body']

# 3. 输出这些行的 body
print(f"共 {len(df_no)} 条满足条件的记录，正文如下：\n")
for i, text in enumerate(df_no, start=1):
    print(f"{i:4d}: {text}")


输出句子的token数量

In [ ]:
from transformers import AutoTokenizer

model_dir = '/home/wangshuo/resource/AIModels/NLP/bart-large-mnli'
# 1. 加载 tokenizer（这里用 bart-large-mnli）
tokenizer = AutoTokenizer.from_pretrained(model_dir)

# 2. 待统计的文本
text = (
    "How many people would stop supporting Biden if his lawyer told the country that "
    "Biden's former personal assistant needed to be brought out at dawn and killed for saying "
    "something that contradicted Biden?  ME! Conservative Christians, why are you still behind "
    "Trump and his legal team? His lawyer is literally calling for the death of a man who "
    "disagrees with Trump. How does that fit into your moral code? Why aren't you speaking out? "
    "Why are you ok with allowing this to continue? You're cheering this on. Absolutely disgusting! "
    "THIS is why 80 million people said it's time for change. Too many of you are ok with this type "
    "of leadership and behavior even though almost all of you claim to be Christians. "
    "#TrumpianChristians #FakeChristians #FakeGod #FakeLove #FakeCompassion #FakeBible #FakeCommandments"
)

# 3. 分词
tokens = tokenizer.tokenize(text)

# 4. 打印 token 数量
print(f"Token 数量: {len(tokens)}")


In [ ]:
import pandas as pd

def analyze_probability_distribution(csv_path: str, column: str = "ML2_oracle2_probability"):
    """
    统计指定 CSV 文件中概率列的区间分布情况（兼容旧版 pandas）。
    
    参数:
        csv_path (str): CSV 文件路径
        column (str): 需要统计的列名，默认 'ML1_proxy4b1_probability'
    
    返回:
        pandas.DataFrame: 各区间的数量和比例
    """
    # 尝试多种编码读取文件，防止乱码
    try:
        df = pd.read_csv(csv_path, encoding='utf-8')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(csv_path, encoding='utf-8-sig')
        except UnicodeDecodeError:
            df = pd.read_csv(csv_path, encoding='latin1')

    # 检查列是否存在
    if column not in df.columns:
        raise ValueError(f"列 '{column}' 不存在，请检查文件列名（可用 df.columns 查看实际列名）。")

    # 去除无效值
    data = pd.to_numeric(df[column], errors='coerce').dropna()

    # 定义区间
    bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    labels = [f"{bins[i]:.1f}-{bins[i+1]:.1f}" for i in range(len(bins)-1)]

    # 分箱统计
    dist = pd.cut(data, bins=bins, labels=labels, include_lowest=True)
    counts = dist.value_counts().sort_index()

    # 计算比例
    distribution_df = pd.DataFrame({
        "区间": counts.index,
        "数量": counts.values,
        "比例(%)": (counts.values / len(data) * 100).round(2)
    })

    print("\n📊 概率分布统计结果：")
    print(distribution_df.to_string(index=False))

    return distribution_df


In [ ]:
result = analyze_probability_distribution(
    "/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/neo4j_data/comment.csv"
    ,column='ML2_proxy2d2_probability'
)
